# 🗺️ Amenity Access Around Transit Stations — 3D Lonboard Map

This notebook visualizes **individual amenity points within a ½-mile (804m) radius** of each BART/Caltrain station using Lonboard.

### Layers
1. **Buildings** — extruded by height, colored by amenity count within ½ mile of the nearest station
2. **Amenity points** — individual points colored by category (park, grocery, clinic, etc.)
3. **Station points** — sized by ridership, colored by agency
4. **½-mile station buffers** — translucent rings showing the catchment area

---
**Data sources**
- `final_station_data.csv` — station locations, amenity counts, ridership
- `all_amenities.csv` — individual amenity points (OSM)
- Overture Maps — SF building footprints (fetched live)

## 1. Install dependencies

In [1]:
!uv pip -q install "geopandas[all]" pyproj lonboard mapclassify ipywidgets overturemaps palettable requests

zsh:1: command not found: uv


## 2. Imports

In [2]:
# Colab only — remove if running locally
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import requests
import overturemaps

from lonboard import Map, SolidPolygonLayer, ScatterplotLayer
from lonboard.basemap import MaplibreBasemap, CartoStyle
from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap
import ipywidgets as W

print('✅ Imports complete')

✅ Imports complete


## 3. Load station & amenity data

In [5]:
# ── Station data ──────────────────────────────────────────────────────────────
stations = pd.read_csv('../data/processed/final_station_data.csv', encoding='latin-1')

stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['longitude'], stations['latitude']),
    crs='EPSG:4326'
)

print(f'Stations loaded: {len(stations_gdf)} ({stations_gdf.agency.value_counts().to_dict()})')
stations_gdf[['station_name', 'agency', 'station_type', 'total_amenities', 'ridership']].head()

Stations loaded: 79 ({'BART': 49, 'Caltrain': 30})


,station_name,agency,station_type,total_amenities,ridership
0,12th St. Oakland City Center,BART,core,53,5164.0
1,16th St. Mission,BART,core,45,5642.0
2,19th St. Oakland,BART,core,39,4606.0
3,24th St. Mission,BART,core,50,5491.0
4,Antioch,BART,peripheral,5,523.0


In [9]:
# ── Amenity points ────────────────────────────────────────────────────────────
amenities = pd.read_csv('../data/raw/all_amenities.csv', index_col=0)

amenities_gdf = gpd.GeoDataFrame(
    amenities,
    geometry=gpd.points_from_xy(amenities['longitude'], amenities['latitude']),
    crs='EPSG:4326'
)

print(f'Amenities loaded: {len(amenities_gdf)}')
print(amenities_gdf['category'].value_counts())

Amenities loaded: 11360
category
park             5541
convenience      2133
grocery          1341
clinic            601
pharmacy          533
doctors           416
kindergartens     338
childcare         296
hospital          161
Name: count, dtype: int64


## 4. Filter amenities to ½-mile of any station

We project to a meter-based CRS, buffer each station by **804 m (½ mile)**, union the buffers, then clip amenities to that footprint.

In [10]:
HALF_MILE_M = 804  # metres

# Project to California Albers (meters) for accurate buffering
stations_m  = stations_gdf.to_crs('EPSG:3310')
amenities_m = amenities_gdf.to_crs('EPSG:3310')

# Per-station buffers
station_buffers = stations_m.copy()
station_buffers['geometry'] = stations_m.geometry.buffer(HALF_MILE_M)

# Spatial join: tag each amenity with the station(s) it falls inside
amenities_near = gpd.sjoin(
    amenities_m,
    station_buffers[['station_name', 'agency', 'station_type', 'ridership', 'geometry']],
    how='inner',
    predicate='within'
).drop(columns='index_right')

# If an amenity is within range of multiple stations, keep the nearest one
amenities_near = amenities_near.loc[
    amenities_near.groupby(amenities_near.index)['station_name']
    .transform('first') == amenities_near['station_name']
].drop_duplicates()

# Back to WGS84 for mapping
amenities_near = amenities_near.to_crs('EPSG:4326')

print(f'Amenities within ½ mile of a station: {len(amenities_near):,} / {len(amenities_gdf):,}')
print(amenities_near['category'].value_counts())

Amenities within ½ mile of a station: 1,105 / 11,360
category
convenience      341
park             331
grocery          151
clinic           100
pharmacy          62
doctors           56
kindergartens     34
childcare         18
hospital          12
Name: count, dtype: int64


## 5. Fetch SF building footprints from Overture Maps

In [11]:
# SF bounding box
bbox = -122.5145, 37.7085, -122.3716, 37.8092

# Get the latest Overture release
latest_release = requests.get('https://stac.overturemaps.org/catalog.json').json().get('latest')
print(f'Overture release: {latest_release}')

# Fetch buildings — this may take a minute
buildings_gdf = (
    overturemaps.core.geodataframe(
        'building',
        bbox=bbox,
        release=latest_release,
    )
)
buildings_gdf = buildings_gdf.set_crs(4326)

print(f'Buildings fetched: {len(buildings_gdf):,}')

Overture release: 2026-03-18.0
Buildings fetched: 161,865


In [12]:
# Clean height column
buildings_gdf['height_clean'] = (
    buildings_gdf['height']
    .fillna(4.0)          # default ~1-storey if missing
    .clip(lower=1.0)
)

print('Height stats:')
print(buildings_gdf['height_clean'].describe().round(1))

Height stats:
count    161865.0
mean          7.9
std           5.2
min           1.0
25%           6.0
50%           7.0
75%           9.0
max         326.0
Name: height_clean, dtype: float64


## 6. Join amenity counts onto buildings

For each building, find the nearest station and attach that station's **total amenity count within ½ mile**.

In [13]:
# Project buildings & stations to meters for the spatial join
buildings_m   = buildings_gdf[['height_clean', 'geometry']].to_crs('EPSG:3310')
stations_m    = stations_gdf[['station_name', 'total_amenities', 'ridership',
                               'agency', 'station_type', 'geometry']].to_crs('EPSG:3310')

# Only keep SF stations (BART within the SF bbox)
sf_stations_m = stations_m[stations_m.geometry.within(
    gpd.GeoSeries(
        [__import__('shapely').geometry.box(-122.5145, 37.7085, -122.3716, 37.8092)],
        crs='EPSG:4326'
    ).to_crs('EPSG:3310').iloc[0]
)]

print(f'SF stations in bbox: {len(sf_stations_m)}')
print(sf_stations_m['station_name'].tolist())

SF stations in bbox: 10
['16th St. Mission', '24th St. Mission', 'Balboa Park', 'Civic Center/UN Plaza', 'Embarcadero', 'Glen Park', 'Montgomery St.', 'Powell St.', '22nd Street', 'San Francisco Caltrain Station']


In [14]:
# Buffer SF stations by ½ mile, then spatial join to buildings
sf_buffers = sf_stations_m.copy()
sf_buffers['geometry'] = sf_stations_m.geometry.buffer(HALF_MILE_M)

buildings_joined = gpd.sjoin(
    buildings_m,
    sf_buffers[['station_name', 'total_amenities', 'ridership', 'agency', 'geometry']],
    how='left',
    predicate='intersects'
).drop(columns='index_right')

# If building touches multiple station buffers, keep the highest amenity count
buildings_joined = (
    buildings_joined
    .sort_values('total_amenities', ascending=False)
    .groupby(buildings_joined.index)
    .first()
)

buildings_joined['total_amenities'] = buildings_joined['total_amenities'].fillna(0)
buildings_joined = buildings_joined.to_crs('EPSG:4326')

print(f'Buildings with amenity data: {(buildings_joined.total_amenities > 0).sum():,} / {len(buildings_joined):,}')
print('Amenity count distribution:')
print(buildings_joined['total_amenities'].describe().round(1))

ValueError: Cannot transform naive geometries.  Please set a crs on the object first.

## 7. Build color arrays

### 7a. Buildings — colored by amenity count (sequential)

In [ ]:
from matplotlib.colors import Normalize
from palettable.colorbrewer.sequential import YlOrRd_9

amenity_vals = buildings_joined['total_amenities'].to_numpy(dtype=float)

# Normalize 0 → max count
norm = Normalize(vmin=0, vmax=amenity_vals.max(), clip=True)
normed = norm(amenity_vals)

building_colors = apply_continuous_cmap(normed, YlOrRd_9, alpha=200)

print(f'Building color array shape: {building_colors.shape}')

### 7b. Amenity points — colored by category

In [ ]:
from palettable.cartocolors.qualitative import Safe_10

# Only SF amenities (filter by bbox)
sf_amenities = amenities_near.cx[-122.5145:-122.3716, 37.7085:37.8092].copy()

CATEGORY_ORDER = [
    'park', 'grocery', 'convenience', 'clinic',
    'pharmacy', 'doctors', 'hospital', 'childcare', 'kindergartens'
]

palette = (np.array(Safe_10.mpl_colors[:len(CATEGORY_ORDER)]) * 255).astype(np.uint8)
cat_color_map = {cat: palette[i].tolist() for i, cat in enumerate(CATEGORY_ORDER)}

sf_amenities['category'] = (
    sf_amenities['category']
    .astype('string')
    .fillna('park')
)

amenity_categorical = sf_amenities['category'].astype(
    pd.CategoricalDtype(categories=CATEGORY_ORDER)
)
amenity_colors = apply_categorical_cmap(amenity_categorical, cat_color_map, alpha=220)

print(f'SF amenity points: {len(sf_amenities):,}')
print(sf_amenities['category'].value_counts())

### 7c. Stations — colored by agency, sized by ridership

In [ ]:
sf_stations_wgs = stations_gdf[stations_gdf.geometry.within(
    __import__('shapely').geometry.box(-122.5145, 37.7085, -122.3716, 37.8092)
)].copy()

# Agency colors: BART=blue, Caltrain=red
AGENCY_COLORS = {'BART': [0, 100, 200, 240], 'Caltrain': [200, 50, 50, 240]}
station_colors = np.array(
    [AGENCY_COLORS.get(a, [120, 120, 120, 200]) for a in sf_stations_wgs['agency']],
    dtype=np.uint8
)

# Radius scaled by ridership (min 80m, max 300m)
ridership = sf_stations_wgs['ridership'].fillna(sf_stations_wgs['ridership'].median()).to_numpy()
r_norm = (ridership - ridership.min()) / (ridership.max() - ridership.min() + 1e-9)
station_radii = (r_norm * 220 + 80).astype(float)

print(f'SF stations: {len(sf_stations_wgs)}')

## 8. Build station buffer rings (½-mile catchment)

In [ ]:
from lonboard import PolygonLayer

buffers_wgs = sf_stations_wgs.copy().to_crs('EPSG:3310')
buffers_wgs['geometry'] = buffers_wgs.geometry.buffer(HALF_MILE_M)
buffers_wgs = buffers_wgs.to_crs('EPSG:4326')

buffer_colors = np.array(
    [AGENCY_COLORS.get(a, [120, 120, 120, 200]) for a in buffers_wgs['agency']],
    dtype=np.uint8
)
# Make fills very transparent — just show the ring outline effect
buffer_fill = buffer_colors.copy()
buffer_fill[:, 3] = 18  # very low alpha

## 9. Assemble the map

In [ ]:
try:
    from lonboard.basemap import MaplibreBasemap, CartoStyle
    map_kwargs = {'basemap': MaplibreBasemap(style=CartoStyle.DarkMatter)}
except (ImportError, AttributeError):
    map_kwargs = {'basemap_style': 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'}

# ── Layer 1: Buildings (3D, colored by amenity count) ──────────────────────────
buildings_layer = SolidPolygonLayer.from_geopandas(
    buildings_joined[['height_clean', 'total_amenities', 'station_name', 'geometry']],
    extruded=True,
    get_elevation=buildings_joined['height_clean'].to_numpy(),
    elevation_scale=1.0,
    get_fill_color=building_colors,
    get_line_color=[20, 20, 20, 40],
    wireframe=False,
    pickable=True,
)

# ── Layer 2: ½-mile buffer rings ───────────────────────────────────────────────
buffer_layer = PolygonLayer.from_geopandas(
    buffers_wgs,
    get_fill_color=buffer_fill,
    get_line_color=buffer_colors,
    line_width_min_pixels=1.5,
    pickable=False,
)

# ── Layer 3: Amenity points ────────────────────────────────────────────────────
amenity_layer = ScatterplotLayer.from_geopandas(
    sf_amenities,
    get_fill_color=amenity_colors,
    get_radius=55,
    radius_min_pixels=3,
    radius_max_pixels=10,
    pickable=True,
    opacity=0.85,
)

# ── Layer 4: Station points ────────────────────────────────────────────────────
station_layer = ScatterplotLayer.from_geopandas(
    sf_stations_wgs,
    get_fill_color=station_colors,
    get_radius=station_radii,
    radius_min_pixels=6,
    radius_max_pixels=22,
    get_line_color=[255, 255, 255, 200],
    line_width_min_pixels=1,
    stroked=True,
    pickable=True,
    opacity=1.0,
)

print('✅ Layers built')

## 10. Legend widget

In [ ]:
def make_legend(title, items, note=''):
    """items: list of (label, hex_color) tuples"""
    rows = ''.join(
        f'<div style="display:flex;align-items:center;margin:3px 0">'
        f'<span style="width:12px;height:12px;border-radius:3px;background:{c};'
        f'border:1px solid #fff3;margin-right:8px;flex-shrink:0"></span>'
        f'<span style="font-size:11px;color:#ddd">{l}</span></div>'
        for l, c in items
    )
    note_html = f'<p style="font-size:10px;color:#aaa;margin-top:6px">{note}</p>' if note else ''
    html = f"""
    <div style="background:#1a1a2e;border:1px solid #fff2;border-radius:10px;
                padding:10px 14px;font-family:system-ui,sans-serif;margin-bottom:6px">
      <div style="font-weight:700;font-size:11px;text-transform:uppercase;
                  letter-spacing:.05em;color:#eee;margin-bottom:8px">{title}</div>
      {rows}{note_html}
    </div>"""
    return W.HTML(value=html)


def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*rgb[:3])


amenity_legend = make_legend(
    'Amenity type',
    [(cat, rgb_to_hex(cat_color_map[cat])) for cat in CATEGORY_ORDER]
)

agency_legend = make_legend(
    'Station (size = ridership)',
    [('BART', '#0064c8'), ('Caltrain', '#c83232')],
    note='Ring = ½-mile (804m) catchment'
)

building_legend = make_legend(
    'Buildings — amenities within ½ mi',
    [('Low', '#ffffb2'), ('Medium', '#fd8d3c'), ('High', '#bd0026')],
    note='Height = actual building height'
)

legend_box = W.HBox([building_legend, amenity_legend, agency_legend])
legend_box

## 11. Render the map

In [ ]:
m = Map(
    layers=[buildings_layer, buffer_layer, amenity_layer, station_layer],
    height=750,
    **map_kwargs
)

m.set_view_state(
    longitude=-122.419,
    latitude=37.775,
    zoom=13,
    pitch=55,
    bearing=-15,
)

W.VBox([legend_box, m])

## 12. Export to standalone HTML

In [ ]:
m.set_view_state(longitude=-122.419, latitude=37.775, zoom=12, pitch=55, bearing=-15)
m.to_html('sf_amenity_station_map.html', title='SF Amenities within ½ mile of Transit Stations')
print('✅ Saved to sf_amenity_station_map.html')

---
## Appendix: Filter by amenity category

Use this cell to quickly re-render showing only one category of amenity.

In [ ]:
# ── Change this to any category in CATEGORY_ORDER ────────────────────────────
FILTER_CATEGORY = 'grocery'   # e.g. 'park', 'clinic', 'pharmacy', etc.

filtered = sf_amenities[sf_amenities['category'] == FILTER_CATEGORY].copy()
filtered_colors = np.tile(
    np.array(cat_color_map[FILTER_CATEGORY] + [230], dtype=np.uint8),
    (len(filtered), 1)
)

filtered_layer = ScatterplotLayer.from_geopandas(
    filtered,
    get_fill_color=filtered_colors,
    get_radius=80,
    radius_min_pixels=4,
    radius_max_pixels=14,
    pickable=True,
    opacity=0.9,
)

m_filtered = Map(
    layers=[buildings_layer, buffer_layer, filtered_layer, station_layer],
    height=700,
    **map_kwargs
)
m_filtered.set_view_state(longitude=-122.419, latitude=37.775, zoom=13, pitch=55, bearing=-15)

title_widget = W.HTML(
    f'<div style="background:#1a1a2e;color:#eee;padding:8px 14px;border-radius:8px;'
    f'font-family:system-ui;font-size:13px;margin-bottom:6px">'
    f'Showing: <b>{FILTER_CATEGORY}</b> ({len(filtered):,} locations)</div>'
)
W.VBox([title_widget, m_filtered])